# Auditoría y Perfilado de Datos Crudos (E-Commerce)

En este notebook realizamos el diagnóstico inicial de calidad de datos sobre los extractos en `datos/raw/`:
1. Perfilado por columna y por tabla (nulos, duplicados, cardinalidad y formatos sospechosos).
2. Verificación de integridad referencial (claves foráneas y registros huérfanos).
3. Validaciones de reglas de negocio (rangos numéricos y coherencia de fechas).

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd

# Definir rutas base del proyecto de forma dinámica
PROJECT_PATH = Path.cwd().resolve().parent
DATA_PATH = PROJECT_PATH / "datos"
RAW_DATA_PATH = DATA_PATH / "raw"

# Cargar las 7 tablas crudas en un diccionario centralizado
dfs = {
    "campañas":      pd.read_csv(RAW_DATA_PATH / "campana.csv"),
    "clientes":      pd.read_csv(RAW_DATA_PATH / "clientes.csv"),
    "devoluciones":  pd.read_csv(RAW_DATA_PATH / "devoluciones.csv"),
    "lineas_pedido": pd.read_csv(RAW_DATA_PATH / "lineas_pedido.csv"),
    "pedidos":       pd.read_csv(RAW_DATA_PATH / "pedidos.csv"),
    "productos":     pd.read_csv(RAW_DATA_PATH / "productos.csv"),
    "visitas_web":   pd.read_csv(RAW_DATA_PATH / "visitas_web.csv"),
}

# Referencias rápidas por tabla para validaciones puntuales
campanas = dfs["campañas"]
clientes = dfs["clientes"]
devoluciones = dfs["devoluciones"]
lineas_pedido = dfs["lineas_pedido"]
pedidos = dfs["pedidos"]
productos = dfs["productos"]
visitas_web = dfs["visitas_web"]

## 1. Funciones de Perfilado (Columna y Tabla)

In [14]:
def profile_column(s: pd.Series, nombre: str) -> dict:
    """Resume calidad, cardinalidad, rangos y posibles anomalías de formato en una serie."""
    n = len(s)
    nulos = int(s.isna().sum())
    no_nulos = s.dropna()
    unicos = int(s.nunique(dropna=True))

    info = {
        "columna": nombre,
        "tipo_pandas": str(s.dtype),
        "nulos": nulos,
        "pct_nulos": round(100 * nulos / n, 2) if n else 0.0,
        "unicos": unicos,
        "pct_unicos": round(100 * unicos / n, 2) if n else 0.0,
        "ejemplo": no_nulos.iloc[0] if len(no_nulos) else None,
    }

    # Identifica columnas compuestas únicamente por dígitos (candidatas a ID)
    if len(no_nulos) and no_nulos.astype(str).str.fullmatch(r"\d+").all():
        info["posible_id"] = True
        info["id_unico"] = unicos == len(no_nulos)

    # Estadísticas descriptivas para columnas numéricas
    if pd.api.types.is_numeric_dtype(s):
        info.update({
            "min": no_nulos.min() if len(no_nulos) else None,
            "max": no_nulos.max() if len(no_nulos) else None,
            "media": round(float(no_nulos.mean()), 2) if len(no_nulos) else None,
            "mediana": no_nulos.median() if len(no_nulos) else None,
            "std": round(float(no_nulos.std()), 2) if len(no_nulos) > 1 else None,
        })

    # Métricas y heurísticas de calidad para columnas de texto / objeto
    elif pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
        longitudes = no_nulos.astype(str).str.len()
        info.update({
            "long_min": int(longitudes.min()) if len(longitudes) else None,
            "long_max": int(longitudes.max()) if len(longitudes) else None,
            "vacio_o_espacios": int(no_nulos.astype(str).str.strip().eq("").sum()),
        })

        # Revisión sobre una muestra para detectar inconsistencias comunes en CSV/Sheets
        muestra = no_nulos.astype(str).head(200)
        if muestra.str.contains(r"\d{4}-\d{2}-\d{2}").any() and muestra.str.contains(r"\d{2}/\d{2}/\d{4}").any():
            info["formato_mixto_fechas"] = True
        if muestra.str.contains(r"€|,|\$").any():
            info["posible_importe_texto"] = True
        if muestra.str.contains(r"%").any():
            info["posible_porcentaje_texto"] = True
        if muestra.str.match(r"^[a-záéíóúñ]").any() and muestra.str.match(r"^[A-ZÁÉÍÓÚÑ]").any():
            info["mayus_minus_mezcladas"] = True

    return info


def profile_table(df: pd.DataFrame, nombre_tabla: str) -> dict:
    """Imprime un diagnóstico completo de un DataFrame y devuelve un diccionario con los resultados."""
    print(f"\n{'=' * 60}\nTABLA: {nombre_tabla.upper()}\n{'=' * 60}")
    print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")

    # Filas idénticas en todas las columnas
    dup_total = int(df.duplicated().sum())
    print(f"Filas totalmente duplicadas: {dup_total:,}")

    # Perfil detallado de cada columna
    perfil_cols = pd.DataFrame([profile_column(df[c], c) for c in df.columns])
    print("\nResumen columnas:")
    print(perfil_cols.to_string(index=False))

    # Revisa duplicidad en la primera columna terminada en '_id' como candidata a PK
    pk = next((c for c in df.columns if c.endswith("_id")), None)
    if pk:
        dup_pk = int(df[pk].duplicated(keep=False).sum())
        print(f"\nColumna PK candidata: {pk}")
        print(f"  IDs únicos: {df[pk].nunique():,} de {len(df):,}")
        print(f"  Filas con ID repetido: {dup_pk:,} ({100 * dup_pk / len(df):.2f}%)")

    # Resumen estadístico general
    print("\nEstadísticas descriptivas:")
    print(df.describe(include="all").T.to_string())

    return {
        "tabla": nombre_tabla,
        "filas": len(df),
        "columnas": df.shape[1],
        "duplicados_total": dup_total,
        "pk_candidata": pk,
        "perfil_columnas": perfil_cols,
    }

In [15]:
# Perfilar cada tabla y guardar su nombre en .attrs para los reportes de FK
resultados = {}
for nombre, df in dfs.items():
    df.attrs["nombre"] = nombre
    resultados[nombre] = profile_table(df, nombre)


TABLA: CAMPAÑAS
Filas: 4 | Columnas: 8
Filas totalmente duplicadas: 0

Resumen columnas:
            columna tipo_pandas  nulos  pct_nulos  unicos  pct_unicos             ejemplo posible_id id_unico      min      max    media  mediana       std  long_min  long_max  vacio_o_espacios posible_importe_texto
         campana_id       int64      0        0.0       4       100.0                   1       True     True      1.0      4.0      2.5      2.5      1.29       NaN       NaN               NaN                   NaN
             nombre         str      0        0.0       4       100.0 Rebajas Verano 2024        NaN      NaN      NaN      NaN      NaN      NaN       NaN      17.0      21.0               0.0                   NaN
               tipo         str      0        0.0       3        75.0             rebajas        NaN      NaN      NaN      NaN      NaN      NaN       NaN       5.0      12.0               0.0                   NaN
       fecha_inicio         str      0        

## 2. Verificación de Integridad Referencial (Foreign Keys)

In [19]:
def check_fk(
    df_hijo: pd.DataFrame,
    col_fk: str,
    df_padre: pd.DataFrame,
    col_pk: str,
    nombre_rel: str,
) -> int:
    """Verifica cuántos valores de una clave foránea no existen en la tabla padre."""
    valores_padre = set(df_padre[col_pk].dropna().unique())
    huerfanos = df_hijo[~df_hijo[col_fk].isin(valores_padre)]
    n_huerf = len(huerfanos)
    pct = 100 * n_huerf / len(df_hijo) if len(df_hijo) else 0.0

    nombre_hijo = df_hijo.attrs.get("nombre", "hijo")
    nombre_padre = df_padre.attrs.get("nombre", "padre")

    print(f"\n{nombre_rel}: {nombre_hijo}.{col_fk} -> {nombre_padre}.{col_pk}")
    print(f"  Huérfanos: {n_huerf:,} ({pct:.2f}%)")
    if n_huerf:
        print(f"  Ejemplos: {huerfanos[col_fk].head(5).tolist()}")
    return n_huerf


print("\n" + "=" * 60 + "\nINTEGRIDAD REFERENCIAL\n" + "=" * 60)
check_fk(pedidos, "cliente_id", clientes, "cliente_id", "pedidos -> clientes")
check_fk(lineas_pedido, "pedido_id", pedidos, "pedido_id", "lineas_pedido -> pedidos")
check_fk(lineas_pedido, "producto_id", productos, "producto_id", "lineas_pedido -> productos")
check_fk(devoluciones, "linea_id", lineas_pedido, "linea_id", "devoluciones -> lineas_pedido")
check_fk(
    pedidos.dropna(subset=["campana_id"]),
    "campana_id",
    campanas,
    "campana_id",
    "pedidos -> campañas"
)


INTEGRIDAD REFERENCIAL

pedidos -> clientes: pedidos.cliente_id -> clientes.cliente_id
  Huérfanos: 0 (0.00%)

lineas_pedido -> pedidos: lineas_pedido.pedido_id -> pedidos.pedido_id
  Huérfanos: 0 (0.00%)

lineas_pedido -> productos: lineas_pedido.producto_id -> productos.producto_id
  Huérfanos: 0 (0.00%)

devoluciones -> lineas_pedido: devoluciones.linea_id -> lineas_pedido.linea_id
  Huérfanos: 42 (0.76%)
  Ejemplos: [2034641, 2011474, 2027240, 2046423, 2014402]

pedidos -> campañas: pedidos.campana_id -> campañas.campana_id
  Huérfanos: 0 (0.00%)


0

## 3. Validaciones de Reglas de Negocio (Rangos y Fechas)

In [23]:
def validate_range(df: pd.DataFrame, col: str, minimo=None, maximo=None) -> pd.DataFrame:
    """Detecta registros numéricos fuera del intervalo [minimo, maximo]."""
    serie = pd.to_numeric(df[col], errors="coerce")
    mask = pd.Series(False, index=df.index)
    if minimo is not None:
        mask |= serie < minimo
    if maximo is not None:
        mask |= serie > maximo

    n_fuera = int(mask.sum())
    pct = 100 * n_fuera / len(df) if len(df) else 0.0
    print(f"{col}: {n_fuera:,} valores fuera de [{minimo}, {maximo}] ({pct:.2f}%)")
    return df[mask]


def validate_dates(df: pd.DataFrame, col_ini: str, col_fin: str) -> pd.DataFrame:
    """Detecta filas donde la fecha de fin es anterior a la fecha de inicio."""
    ini = pd.to_datetime(df[col_ini], errors="coerce")
    fin = pd.to_datetime(df[col_fin], errors="coerce")
    invalidas = df[fin < ini]
    pct = 100 * len(invalidas) / len(df) if len(df) else 0.0
    print(f"{col_fin} < {col_ini}: {len(invalidas):,} filas ({pct:.2f}%)")
    return invalidas


print("\n" + "=" * 60 + "\nVALIDACIONES\n" + "=" * 60)
validate_range(clientes, "edad", 0, 100)
validate_range(lineas_pedido, "cantidad", 1, np.inf)
validate_dates(campanas, "fecha_inicio", "fecha_fin")


VALIDACIONES
edad: 13 valores fuera de [0, 100] (0.26%)
cantidad: 15 valores fuera de [1, inf] (0.07%)
fecha_fin < fecha_inicio: 0 filas (0.00%)


,campana_id,nombre,tipo,fecha_inicio,fecha_fin,canal,presupuesto,ventas_ano_anterior
